<a href="https://colab.research.google.com/github/saikttech/colab-aiagent/blob/main/%D0%94%D0%97_PRO_%D0%9D%D0%B5%D0%B9%D1%80%D0%BE_%D0%BA%D0%BE%D0%BD%D1%81%D1%83%D0%BB%D1%8C%D1%82%D0%B0%D0%BD%D1%82_%D0%BD%D0%B0_%D0%B1%D0%B0%D0%B7%D0%B5_GPT_%D1%81_%D0%B4%D0%B8%D0%B0%D0%BB%D0%BE%D0%B3%D0%BE%D0%BC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Напишите функцию, которая принимает ввод от пользователя и возвращает ответ от модели GPT, а также подсчитывает количество токенов любым способом. Ваша функция должна быть способна обрабатывать любой ввод пользователя. Можно использовать любую модель GPT.

# Решение

In [ ]:
import requests
import json
from google.colab import userdata
import ipywidgets as widgets
from IPython.display import display, clear_output

# ==========================================
# Функция для работы с Yandex GPT API
# ==========================================
def get_yandex_gpt_response(user_input, model="yandexgpt-lite"):
    """
    Отправляет запрос к Yandex GPT и возвращает ответ с подсчетом токенов.
    """
    try:
        oauth_token = userdata.get('YC_API_KEY')
        folder_id = userdata.get('YC_FOLDER_ID')
    except userdata.SecretNotFoundError as e:
        return None, None, f"Ошибка: {e}. Проверьте названия секретов в Colab."

    prompt_text = str(user_input)

    url = "https://llm.api.cloud.yandex.net/foundationModels/v1/completion"
    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {oauth_token}"
    }

    payload = {
        "modelUri": f"gpt://{folder_id}/{model}",
        "completionOptions": {
            "stream": False,
            "temperature": 0.6,
            "maxTokens": "2000"
        },
        "messages": [
            {
                "role": "system",
                "text": "Ты полезный и вежливый ассистент. Отвечай четко и по делу."
            },
            {
                "role": "user",
                "text": prompt_text
            }
        ]
    }

    try:
        response = requests.post(url, headers=headers, json=payload)
        response.raise_for_status()

        data = response.json()
        answer = data['result']['alternatives'][0]['message']['text']

        usage = data['result']['usage']
        tokens_info = {
            "input_tokens": int(usage['inputTextTokens']),
            "output_tokens": int(usage['completionTokens']),
            "total_tokens": int(usage['totalTokens'])
        }

        return answer, tokens_info, None

    except requests.exceptions.HTTPError as http_err:
        return None, None, f"HTTP ошибка: {http_err}\nОтвет сервера: {response.text}"
    except Exception as err:
        return None, None, f"Произошла ошибка: {err}"

# ==========================================
# Создание виджетов интерфейса
# ==========================================

# Поле для ввода запроса
input_text = widgets.Textarea(
    value='',
    placeholder='Введите ваш запрос к Yandex GPT...',
    description='Запрос:',
    disabled=False,
    layout=widgets.Layout(width='100%', height='100px')
)

# Выбор модели
model_dropdown = widgets.Dropdown(
    options=[
        ('YandexGPT Lite (быстрая)', 'yandexgpt-lite'),
        ('YandexGPT (умная)', 'yandexgpt'),
        ('YandexGPT 32K (длинный контекст)', 'yandexgpt-32k')
    ],
    value='yandexgpt-lite',
    description='Модель:',
    disabled=False,
    layout=widgets.Layout(width='400px')
)

# Кнопка отправки
send_button = widgets.Button(
    description='Отправить запрос',
    disabled=False,
    button_style='primary',
    tooltip='Нажмите для отправки запроса к Yandex GPT',
    icon='paper-plane'
)

# Область для вывода результата
output_area = widgets.Output(
    layout=widgets.Layout(
        width='100%',
        min_height='200px',
        border='1px solid #ddd',
        padding='10px',
        margin='10px 0px'
    )
)

# Статистика токенов
tokens_label = widgets.HTML(
    value='<b>Статистика токенов:</b><br>Ожидание запроса...',
    layout=widgets.Layout(width='100%', padding='10px', background_color='#f0f0f0')
)

# ==========================================
# Обработчик кнопки
# ==========================================
def on_send_button_clicked(b):
    # Блокируем кнопку на время запроса
    send_button.disabled = True
    send_button.description = 'Обработка...'
    send_button.icon = 'spinner'

    # Очищаем предыдущий вывод
    output_area.clear_output()

    # Получаем данные из виджетов
    user_query = input_text.value.strip()
    selected_model = model_dropdown.value

    if not user_query:
        with output_area:
            print("⚠️ Пожалуйста, введите запрос.")
        send_button.disabled = False
        send_button.description = 'Отправить запрос'
        send_button.icon = 'paper-plane'
        return

    # Отображаем индикатор загрузки
    with output_area:
        print("⏳ Генерирую ответ... Пожалуйста, подождите.")

    # Вызываем функцию API
    answer, tokens, error = get_yandex_gpt_response(user_query, selected_model)

    # Очищаем индикатор загрузки
    output_area.clear_output()

    # Отображаем результат
    with output_area:
        if error:
            print(f"❌ {error}")
            tokens_label.value = '<b>Статистика токенов:</b><br>Ошибка выполнения запроса'
        else:
            print(f"🤖 <b>Ответ Yandex GPT:</b>\n")
            print(answer)

            # Обновляем статистику токенов
            tokens_label.value = f"""
            <b>📊 Статистика токенов:</b><br>
            • Токенов в запросе (input): <b>{tokens['input_tokens']}</b><br>
            • Токенов в ответе (output): <b>{tokens['output_tokens']}</b><br>
            • Всего токенов (total): <b>{tokens['total_tokens']}</b>
            """

    # Разблокируем кнопку
    send_button.disabled = False
    send_button.description = 'Отправить запрос'
    send_button.icon = 'paper-plane'

# Привязываем обработчик к кнопке
send_button.on_click(on_send_button_clicked)

# ==========================================
# Отображение интерфейса
# ==========================================
print("🚀 Интерфейс Yandex GPT")
print("=" * 50)

# Создаем контейнер для всех виджетов
ui_container = widgets.VBox([
    widgets.HTML("<h3>Введите ваш запрос:</h3>"),
    input_text,
    widgets.HBox([model_dropdown, send_button]),
    output_area,
    tokens_label
])

display(ui_container)

🚀 Интерфейс Yandex GPT
